In [ ]:
# Autoload when refreshing notebook
%load_ext autoreload
%autoreload 2

import numpy as np
import h5py
from scipy.io import loadmat
import pandas as pd
import re
import matplotlib.pyplot as plt
from types import SimpleNamespace
import scipy
import warnings
from scipy.ndimage import median_filter, gaussian_filter
from scipy.optimize import curve_fit

# import Python functions 
import sys
sys.path.append('../../')

from Python_Functions.functions import matstruct_to_dict, extractDAQBSAScalars, extractDAQNonBSAScalars, apply_tcav_zeroing_filter, analyze_SYAG, facet_daq_path, extract_UVVisSpec, exclude_bsa_vars, sanitize_FACET_input 
from Python_Functions.gmm import biGaussian_image_from_flattened_params, flatten_biGaussian_params, unflatten_biGaussian_params

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import pickle
import numpy as np
from scipy.io import loadmat 
import re
import os 
import joblib
# Assumed: commonIndexFromSteps, extractDAQBSAScalars, and other helper functions are available

# ----------------------------------------------------------------------
# 0. Load the model with joblib
# Ex: MLP_LPS_GMM_E338_12710_20251031_194740


#joblib_file = '../../model/LPS/MLP_LPS_GMM_Forest_E338_12710_20251206_154313.pkl'  # Modify as needed
#model = joblib.load(joblib_file)
#iz_scaler = pickle.load(open('../../model/LPS/E338_12710_iz_scaler_GMM_20251206_154313.pkl', 'rb'))
# ----------------------------------------------------------------------
# 1. Define the list of (experiment, runname, step_identifier) pairs to test the model on.
# ----------------------------------------------------------------------
run_pairs = [
    ('E338', '16269', 1),  # Example pairs, modify this list
    #('E338', '16266', 1),
    # Add more pairs here...
]

# ----------------------------------------------------------------------
# 2. Initialize lists for concatenation
# ----------------------------------------------------------------------
all_predictors = []
all_indices = []

print("Starting multi-run data loading and concatenation...")

# ----------------------------------------------------------------------
# 3. Loop through runs, load data, and concatenate
# ----------------------------------------------------------------------
for experiment, runname, step_id in run_pairs:
        
    # --- B. Load and Filter Predictor Data (BSA Scalars) ---
    
    # 1. Load data_struct
    dataloc = facet_daq_path(experiment = experiment, daq_num = runname)
    try:
        mat = loadmat(dataloc,struct_as_record=False, squeeze_me=True)
        data_struct = mat['data_struct']
    except FileNotFoundError:
        print(f"Skipping {experiment}_{runname}: .mat file not found at {dataloc}")
        continue

    # 2. Extract full BSA scalars (filtered by step_list if needed)
    # Don't filter by common index here, we'll do it with the goodShots scalar common index loaded from the file
    bsaScalarData, bsaVars = extractDAQBSAScalars(data_struct, filter_index=True)
    nonBsaScalarData, nonBsaVars = extractDAQNonBSAScalars(data_struct, filter_index=True)
    bsaScalarData = apply_tcav_zeroing_filter(bsaScalarData, bsaVars)

    ampl_idx = next(i for i, var in enumerate(bsaVars) if 'TCAV_LI20_2400_A' in var)
    xtcavAmpl = bsaScalarData[ampl_idx, :]

    phase_idx = next(i for i, var in enumerate(bsaVars) if 'TCAV_LI20_2400_P' in var)
    xtcavPhase = bsaScalarData[phase_idx, :]
    xtcavOffShots = xtcavAmpl<0.1
    xtcavPhase[xtcavOffShots] = 0 #Set this for ease of plotting
    fig, ax1 = plt.subplots()
    ax1.plot(xtcavAmpl, label='Amplitude', color='b')
    ax1.set_ylabel('XTCAV Ampl [MV]', color='b')
    ax1.tick_params(axis='y', labelcolor='b')

    ax2 = ax1.twinx()
    ax2.plot(xtcavPhase, label='Phase', color='r')
    ax2.set_ylabel('XTCAV Phase [deg]', color='r')
    ax2.tick_params(axis='y', labelcolor='r')

    plt.title('XTCAV Amplitude and Phase')
    plt.show()
    # 5. Filter BSA data using the final index
    # goodShots_scal_common_index is 1 based indexing from MATLAB, convert to 0 based
    bsaScalarData_filtered = bsaScalarData
    nonBsaScalarData_filtered = nonBsaScalarData
    # 6. Construct the predictor array
    predictor_current = np.vstack(np.concatenate((bsaScalarData_filtered, nonBsaScalarData_filtered),axis = 0)).T
    
    # C. Append to master lists
    all_predictors.append(predictor_current)
    
# ----------------------------------------------------------------------
# 4. Concatenate and finalize arrays
# ----------------------------------------------------------------------
# Combine all data arrays from the runs
predictor_tmp = np.concatenate(all_predictors, axis=0)
all_predictor_names = bsaVars + nonBsaVars
predictor_tmp_cleaned = sanitize_FACET_input(predictor_tmp, all_predictor_names)
excluded_var_idx = exclude_bsa_vars(all_predictor_names, predictor_tmp_cleaned)    
all_predictor_names_simplified = [var for i, var in enumerate(all_predictor_names) if i not in excluded_var_idx]
# Delete the excluded indices from the array
predictor_tmp_simplified = np.delete(predictor_tmp_cleaned, excluded_var_idx, axis=1)
# Set image half dimensions (should match preprocessing)
yrange = 50
xrange = 150
NCOMP = 10  # Number of GMM parameters
print("\n--- Final Concatenated Data Shapes ---")
print(f"Total Predictors (predictor): {predictor_tmp_simplified.shape}")



In [ ]:
from Python_Functions.functions import extract_processed_images
import pathlib

hotPixThreshold = 1e4
sigma = 1
threshold = 50
UVVisSpec_raw = extract_UVVisSpec(data_struct, directory_path=str(pathlib.Path(dataloc).parent))
CHERImages_raw, _, _, _ = extract_processed_images(data_struct, '', None, None, hotPixThreshold, sigma, threshold, step_list=None, roi_xrange=(0, 2000), roi_yrange=(1250, 1600), do_load_raw=False, directory_path=str(pathlib.Path(dataloc).parent), instrument = 'CHER', intermediate_datatype=np.uint16)

In [ ]:
def show_CHER_image(idx):
    plt.figure(figsize=(10, 3))
    plt.plot(UVVisSpec_raw[idx])
    plt.show()
    plt.figure(figsize=(10, 2))
    plt.imshow(CHERImages_raw[:,:,idx],cmap='inferno', vmax = np.percentile(CHERImages_raw[:,:,idx], q=99))
    plt.colorbar(label='Summed Intensity')
    plt.show()
    plt.figure(figsize=(10, 2))
    plt.plot(np.sum(CHERImages_raw[:,:,idx], axis=0))
    plt.show()
# Create slider
import ipywidgets as widgets
from ipywidgets import interact

# Define the slider with a custom layout width
my_slider = widgets.IntSlider(
    min=0, 
    max=UVVisSpec_raw.shape[0]-1, 
    step=1, 
    value=0,
    layout=widgets.Layout(width='100%')  # Change to '800px' for a fixed size
)

interact(show_CHER_image, idx=my_slider)


In [ ]:
# Projection
def show_CHER_proj(idx):
    plt.figure(figsize=(10, 2))
    plt.plot(UVVisSpec_raw[idx, :])
    plt.show()
    plt.figure(figsize=(10, 2))
    plt.plot(np.sum(CHERImages_raw[:,:,idx], axis=0))
    plt.show()
# Create slider
import ipywidgets as widgets
from ipywidgets import interact

# Define the slider with a custom layout width
my_slider = widgets.IntSlider(
    min=0, 
    max=UVVisSpec_raw.shape[0]-1, 
    step=1, 
    value=0,
    layout=widgets.Layout(width='100%')  # Change to '800px' for a fixed size
)

interact(show_CHER_proj, idx=my_slider)


In [ ]:
# Waterfall
def show_CHER_waterfall():
    plt.figure(figsize=(10, 10))
    plt.imshow(UVVisSpec_raw[:, :], vmax = 5000)
    plt.show()
    plt.figure(figsize=(10, 10))
    plt.imshow(np.sum(CHERImages_raw[:,:,:], axis=0), vmin = 0, vmax = 200000)
    plt.show()

show_CHER_waterfall()


In [ ]:
print(data_struct.scalars.steps.shape)
print(data_struct.scalars.common_index.shape)
print(CHERImages_raw.shape)

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

# 1. Collapse the raw images into a profile
cherProfile = np.sum(CHERImages_raw, axis=0)

# 2. Gaussian smooth with a window size (sigma) of 3
# Note: In Gaussian filtering, sigma is the standard deviation. 
# A sigma of 3 roughly corresponds to a physical window of 7-10 pixels.
smoothedProfile = gaussian_filter1d(cherProfile, sigma=3, axis = 0)
goodshots = []
badshots = []
for idx in range(smoothedProfile.shape[1]):
    # 3. Find peaks with the specified constraints
    # height, prominence, and width can all be used, but we'll focus on prominence
    peaks, properties = find_peaks(
        smoothedProfile[:,idx], 
        prominence=20000, 
        height=None  # You can add a minimum height here if needed
    )
    # 4. Filter for peaks between position 500 and 1000
    mask = (peaks >= 700) & (peaks <= 750)
    filtered_peaks = peaks[mask]
    if filtered_peaks.size == 1:
        goodshots.append(idx)
    mask2 = (peaks >= 350) & (peaks <= 1100)
    filtered_peaks2 = peaks[mask2]
    if filtered_peaks2.size == 0:
        badshots.append(idx)
goodshots = np.array(goodshots)
badshots = np.array(badshots)
print(f"Found Good Shots: {goodshots.shape}")
print(f"Found Bad Shots: {badshots.shape}")

In [ ]:
step_to_find = 2

# 1. Get the step values for the shots corresponding to your CHER data
# Assuming common_index matches the order of your CHERImages_raw
shot_steps = data_struct.scalars.steps[data_struct.scalars.common_index - 1]

# 2. Find indices where the step matches
step_idxs = np.where(shot_steps == step_to_find)[0]

# 3. Find the intersection of 'goodshots' and 'step_idxs'
# This gives you shot indices that are BOTH high quality and from the correct step
goodshots_step = np.intersect1d(goodshots, step_idxs)

print(f"Found {len(goodshots_step)} good shots for step {step_to_find}")

# Display the first 5 good shots from this step
for idx in goodshots_step[:10]:
    plt.figure()
    plt.imshow(CHERImages_raw[:,:,idx], vmax = np.percentile(CHERImages_raw[:,:,idx], q=99))
    plt.title(f"Shot Index: {idx} | Step: {step_to_find} | CHER Projection")
    plt.show()
    plt.figure()
    plt.imshow(PAXRADImages_raw[:,:,idx])
    plt.title(f"Shot Index: {idx} | Step: {step_to_find}")
    plt.show()
    plt.figure()
    profile = np.sum(PAXRADImages_raw[20:25,:,idx], axis = 0)
    plt.plot(profile)
    smoothedProfile = gaussian_filter1d(profile, sigma=2, axis = 0)
    peaks, properties = find_peaks(
        smoothedProfile, 
        prominence=400, 
        height=None  # You can add a minimum height here if needed
    )
    for p in peaks:
        plt.axvline(p)
    plt.title(f"Shot Index: {idx} | Step: {step_to_find} | Center Horizontal Profile")
    plt.show()
    plt.plot(np.sum(PAXRADImages_raw[:,128:132,idx], axis = 1))
    plt.title(f"Shot Index: {idx} | Step: {step_to_find} | Witness Vertical Profile")
    plt.show()


In [ ]:
step_to_find = 4

# 1. Get the step values for the shots corresponding to your CHER data
# Assuming common_index matches the order of your CHERImages_raw
shot_steps = data_struct.scalars.steps[data_struct.scalars.common_index - 1]

# 2. Find indices where the step matches
step_idxs = np.where(shot_steps == step_to_find)[0]

# 3. Find the intersection of 'goodshots' and 'step_idxs'
# This gives you shot indices that are BOTH high quality and from the correct step
goodshots_step = np.intersect1d(goodshots, step_idxs)

print(f"Found {len(goodshots_step)} good shots for step {step_to_find}")

# Display the first 5 good shots from this step
for idx in goodshots_step[:10]:
    plt.figure()
    plt.imshow(CHERImages_raw[:,:,idx], vmax = np.percentile(CHERImages_raw[:,:,idx], q=99))
    plt.title(f"Shot Index: {idx} | Step: {step_to_find} | CHER Projection")
    plt.show()
    plt.figure()
    plt.imshow(PAXRADImages_raw[:,:,idx])
    plt.title(f"Shot Index: {idx} | Step: {step_to_find}")
    plt.show()
    plt.figure()
    profile = np.sum(PAXRADImages_raw[55:60,:,idx], axis = 0)
    plt.plot(profile)
    smoothedProfile = gaussian_filter1d(profile, sigma=2, axis = 0)
    peaks, properties = find_peaks(
        smoothedProfile, 
        prominence=400, 
        height=None  # You can add a minimum height here if needed
    )
    for p in peaks:
        plt.axvline(p)
    plt.title(f"Shot Index: {idx} | Step: {step_to_find} | Center Horizontal Profile")
    plt.show()
    plt.plot(np.sum(PAXRADImages_raw[:,141:145,idx], axis = 1))
    plt.title(f"Shot Index: {idx} | Step: {step_to_find} | Witness Vertical Profile")
    plt.show()


In [ ]:
step_to_find = 2

# 1. Get the step values for the shots corresponding to your CHER data
# Assuming common_index matches the order of your CHERImages_raw
shot_steps = data_struct.scalars.steps[data_struct.scalars.common_index - 1]

# 2. Find indices where the step matches
step_idxs = np.where(shot_steps == step_to_find)[0]

# 3. Find the intersection of 'goodshots' and 'step_idxs'
# This gives you shot indices that are BOTH high quality and from the correct step
badshots_step = np.intersect1d(badshots, step_idxs)

print(f"Found {len(badshots_step)} bad shots for step {step_to_find}")

# Display the first 5 good shots from this step
for idx in badshots_step[:10]:
    plt.figure()
    plt.imshow(PAXRADImages_raw[:,:,idx])
    plt.title(f"Shot Index: {idx} | Step: {step_to_find}")
    plt.show()
    plt.figure()
    plt.imshow(CHERImages_raw[:,:,idx], vmax = np.percentile(CHERImages_raw[:,:,idx], q=99))
    plt.title(f"Shot Index: {idx} | Step: {step_to_find} | CHER Projection")
    plt.show()
    plt.figure()
    profile = np.sum(PAXRADImages_raw[55:60,:,idx], axis = 0)
    plt.plot(profile)
    smoothedProfile = gaussian_filter1d(profile, sigma=2, axis = 0)
    peaks, properties = find_peaks(
        smoothedProfile, 
        prominence=400, 
        height=None  # You can add a minimum height here if needed
    )
    for p in peaks:
        plt.axvline(p)
    plt.title(f"Shot Index: {idx} | Step: {step_to_find} | Center Horizontal Profile")
    plt.show()
    plt.plot(np.sum(PAXRADImages_raw[:,98:102,idx], axis = 1))
    plt.title(f"Shot Index: {idx} | Step: {step_to_find} | Witness Vertical Profile")
    plt.show()


## Witness y beta is large

In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d

# Load the pickle file
with open('../../../CurrentProfile/E300_15675_PSDL_CP.pkl', 'rb') as f:
    cp_data = pickle.load(f)

peak_indices = []
separation = []

# --- New Parameters ---
# sigma determines the amount of smoothing. Increase for more aggressive smoothing.
smoothing_sigma = 2.0 
min_peak_distance = 20

for i in range(cp_data["current_profile"].shape[0]):
    raw_profile = cp_data["current_profile"][i]
    
    # 1. Smooth the waveform before finding peaks
    smoothed_profile = gaussian_filter1d(raw_profile, sigma=smoothing_sigma)
    
    # 2. Enforce peaks are at least 20 indices apart using the 'distance' argument
    peaks, properties = find_peaks(
        smoothed_profile, 
        height=0.05 * np.max(smoothed_profile), 
        distance=min_peak_distance
    )
    
    peak_indices.append(peaks)
    
    # Make sure there are exactly two peaks, if not, append NaN
    if len(peaks) == 2:
        separation.append(peaks[1] - peaks[0])
    else:
        separation.append(np.nan)
cal = cp_data["fs_per_px"]
# Plotting the results
separation = np.array(separation, dtype = np.float64)
plt.figure(figsize=(10, 5))
plt.plot(separation * cal * 0.2998, marker='o', markersize=2, linestyle='-', linewidth=0.5)
plt.title('Separation Between Two Peaks 1 Bunch w/ Collimator')
plt.xlabel('Shot Index')
plt.ylabel('Separation (um)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def show_CP_image(idx):
    plt.figure(figsize=(10, 6))
    times = np.linspace(0, cp_data["current_profile"][idx].shape[0] * cp_data["fs_per_px"] * 0.2998,cp_data["current_profile"][idx].shape[0])
    plt.plot(times, cp_data["current_profile"][idx])
    plt.show()
# Create slider
import ipywidgets as widgets
from ipywidgets import interact
interact(show_CP_image, idx=widgets.IntSlider(min=0, max=len(cp_data["current_profile"])-1, step=1, value=0))

In [ ]:
with open("../../Two_Bunch_Training/E300_15675_PSDL_preprocessed.pkl", 'rb') as f:
    data = pickle.load(f)
                
    compare_truth_data = data['LPSImage'].transpose(2, 0, 1)
    truth_cp = np.sum(compare_truth_data, axis = 1)
    phase_class = data['phase_class']
    scalar_common_index = data['scalarCommonIndex']
    xtcalibrationfactor_fs = data['xtcalibrationfactor'] * 1e15
    print("xtcalibrationfactor_fs:", xtcalibrationfactor_fs)
    umPerDeg = data['umPerDeg']
    # This is a 1 based index from MATLAB, which directly corresponds to the shot numbers in the DAQ data. We will use this to align the truth images with the correct DAQ rows for prediction.
    goodShots_scal_common_index_cmp = data['scalarCommonIndex']
    total_charge = 1e10 # electrons


In [ ]:
def show_truth_CP_image(idx):
    plt.figure(figsize=(10, 6))
    times = np.linspace(0, truth_cp[idx].shape[0]*xtcalibrationfactor_fs * 0.29979, truth_cp[idx].shape[0])
    plt.plot(times, truth_cp[idx]/ np.sum(truth_cp[idx]) / xtcalibrationfactor_fs * 1.602e-4 * total_charge)
    plt.show()
# Create slider
import ipywidgets as widgets
from ipywidgets import interact
interact(show_truth_CP_image, idx=widgets.IntSlider(min=0, max=truth_cp.shape[0]-1, step=1, value=0))


In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d

peak_indices_truth = []
separation_truth = []

# --- New Parameters ---
# sigma determines the amount of smoothing. Increase for more aggressive smoothing.
smoothing_sigma = 2.0 
min_peak_distance = 20

for i in range(truth_cp.shape[0]):
    raw_profile = truth_cp[i]
    
    # 1. Smooth the waveform before finding peaks
    smoothed_profile = gaussian_filter1d(raw_profile, sigma=smoothing_sigma)
    
    # 2. Enforce peaks are at least 20 indices apart using the 'distance' argument
    peaks, properties = find_peaks(
        smoothed_profile, 
        height=0.05 * np.max(smoothed_profile), 
        distance=min_peak_distance
    )
    
    peak_indices_truth.append(peaks)
    
    # Make sure there are exactly two peaks, if not, append NaN
    if len(peaks) == 2:
        separation_truth.append(peaks[1] - peaks[0])
    else:
        separation_truth.append(np.nan)
# Plotting the results
separation_truth = np.array(separation_truth, dtype = np.float64)
plt.figure(figsize=(10, 5))
plt.plot(separation_truth[100:] * cal*2/3 * 0.2998, marker='o', markersize=2, linestyle='-', linewidth=0.5, label = 'truth')
plt.plot(separation * cal * 0.2998, marker='o', markersize=2, linestyle='-', linewidth=0.5, label ='prediction')
plt.title('Separation Between Two Peaks 1 Bunch w/ Collimator')
plt.xlabel('Shot Index')
plt.ylabel('Separation (um)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d

# Load the pickle file
with open('../../../CurrentProfile/E300_15675_PSDL_CP.pkl', 'rb') as f:
    cp_data = pickle.load(f)

peak_indices_wrf = []
separation_wrf = []

# --- New Parameters ---
# sigma determines the amount of smoothing. Increase for more aggressive smoothing.
smoothing_sigma = 2.0 
min_peak_distance = 20
indices = range(cp_data["current_profile"].shape[0])
for i in indices:
    raw_profile = cp_data["current_profile"][i]
    
    # 1. Smooth the waveform before finding peaks
    smoothed_profile = gaussian_filter1d(raw_profile, sigma=smoothing_sigma)
    
    # 2. Enforce peaks are at least 20 indices apart using the 'distance' argument
    peaks, properties = find_peaks(
        smoothed_profile, 
        height=0.02 * np.max(smoothed_profile), 
        distance=min_peak_distance
    )
    
    peak_indices_wrf.append(peaks)
    
    # Make sure there are exactly two peaks, if not, append NaN
    if len(peaks) == 2:
        separation_wrf.append(peaks[1] - peaks[0])
    else:
        separation_wrf.append(np.nan)
cal = cp_data["fs_per_px"]
cal = cal * 0.9
# Plotting the results
separation_wrf = np.array(separation_wrf, dtype = np.float64)
plt.figure(figsize=(10, 5))
plt.plot(separation_wrf * cal * 0.2998, marker='o', markersize=2, linestyle='-', linewidth=0.5)
plt.title('Separation Between Two Peaks 1 Bunch w/ Collimator')
plt.xlabel('Shot Index')
plt.ylabel('Separation (um)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d

peak_indices_truth = []
separation_truth = []

# --- New Parameters ---
# sigma determines the amount of smoothing. Increase for more aggressive smoothing.
smoothing_sigma = 2.0 
min_peak_distance = 20

for i in indices2:
    raw_profile = truth_cp[i]
    
    # 1. Smooth the waveform before finding peaks
    smoothed_profile = gaussian_filter1d(raw_profile, sigma=smoothing_sigma)
    
    # 2. Enforce peaks are at least 20 indices apart using the 'distance' argument
    peaks, properties = find_peaks(
        smoothed_profile, 
        height=0.02 * np.max(smoothed_profile), 
        distance=min_peak_distance
    )
    
    peak_indices_truth.append(peaks)
    
    # Make sure there are exactly two peaks, if not, append NaN
    if len(peaks) == 2:
        separation_truth.append(peaks[1] - peaks[0])
    else:
        separation_truth.append(np.nan)
# Plotting the results
separation_truth = np.array(separation_truth, dtype = np.float64)
plt.figure(figsize=(10, 5))
plt.plot(separation_truth[100:] * cal*2/3 * 0.2998, marker='o', markersize=2, linestyle='-', linewidth=0.5, label = 'truth')
plt.plot(separation_wrf * cal * 0.2998, marker='o', markersize=2, linestyle='-', linewidth=0.5, label ='prediction')
print(separation_wrf.shape)
print(separation_truth.shape)
plt.title('Separation Between Two Peaks 1 Bunch w/ Collimator')
plt.xlabel('Shot Index')
plt.ylabel('Separation (um)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
print(predictor_tmp.shape)
print(data_struct.scalars.common_index)
print(scalar_common_index)
# Finds the first matching index in the larger array for each value
# Returns the indices of data_struct.scalars.common_index that match ANY value in scalar_common_index
indices = np.where(np.isin(data_struct.scalars.common_index, scalar_common_index))[0]
indices2 = np.where(np.isin(scalar_common_index, data_struct.scalars.common_index[indices]))[0]

print(indices)

print(indices2)

### 

## Train CVAE1D + Random Forest on UVVis spectra

Reuse the same training routines as `vtcav_display.train_model_cvae_wrf` via
`Python_Functions.training`. The 1-D variant is selected automatically when the
input is 2-D (N, L). The CVAE1D requires `L` to be a multiple of 32, so we
trim/pad the spectrum length first.

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter1d  # Imported for smoothing
from Python_Functions.training import encode_with_cvae, train_random_forest_predictor

# Pick the rows to train on. `goodshots` was identified earlier from the CHER
# peak-finding pass; aligns 1-to-1 with predictor_tmp / UVVisSpec_raw.
sel = np.array([True] * UVVisSpec_raw.shape[0]) #np.asarray(goodshots, dtype=int)
print(f"Selecting {sel.size} good shots out of {UVVisSpec_raw.shape[0]}")

uvvis = UVVisSpec_raw[sel].astype(np.float32)

# Trim/grab the first 2048 samples and drop the extra dimension
L_initial = 2048
uvvis = uvvis[:, :L_initial][:, :, 0]

# 1. Bin and average: 2048 down to 256 (every 8 points averaged into 1)
target_len = 256
bin_size = L_initial // target_len  # 8
uvvis = uvvis.reshape(uvvis.shape[0], target_len, bin_size).mean(axis=2)
print(f"UVVis shape after binning to {target_len}: {uvvis.shape}")

# 2. Apply 1D Gaussian filter along the spectral features (axis 1)
# Note: Adjust sigma=1.0 up or down depending on how much smoothing you want
uvvis = gaussian_filter1d(uvvis, sigma=1.0, axis=1)
print(f"UVVis shape after Gaussian filtering: {uvvis.shape}")

# Normalize per-shot to [0, 1] before encoding (encode_with_cvae also divides
# by the global max, but per-shot makes the latent codes more interpretable).
uvvis_norm = uvvis / np.maximum(uvvis.max(axis=1, keepdims=True), 1e-9)

NCOMP = 16
model_cvae, latent_z = encode_with_cvae(
    uvvis_norm,
    latent_dim=NCOMP,
    n_epochs=10,
    batch_size=32,
    lr=1e-3,
)
print(f"Latent codes shape: {latent_z.shape}")


In [ ]:
# Predictors → latent codes via weighted RandomForest.
# predictor_tmp was built earlier; align with the same `sel` rows.
predictors = predictor_tmp_simplified[sel]
print(f"Predictors: {predictors.shape}, targets: {latent_z.shape}")

rf_result = train_random_forest_predictor(
    predictors,
    latent_z,
    n_estimators=500,
    max_depth=15,
    min_samples_leaf=2,
    test_size=0.2,
    weight_strategy='importance_weighted',
    kde_bandwidth=0.1,
    random_state=42,
    predictor_names=all_predictor_names_simplified
)

rf_model = rf_result['model']
x_scaler = rf_result['x_scaler']
iz_scaler = rf_result['iz_scaler']
print(f"Train R²: {rf_result['train_r2']*100:.2f}%   Test R²: {rf_result['test_r2']*100:.2f}%")

In [ ]:
# Visual sanity check: predicted vs. true UVVis for a few test shots.
import torch

test_idx = rf_result['test_indices'][:6]

x_test_scaled = x_scaler.transform(predictors[test_idx])
latent_pred = iz_scaler.inverse_transform(rf_model.predict(x_test_scaled))

device = next(model_cvae.parameters()).device
with torch.no_grad():
    latent_t = torch.from_numpy(latent_pred).float().to(device)
    recon = model_cvae.decode_latent_mu(latent_t).cpu().numpy().squeeze(1)

fig, axes = plt.subplots(len(test_idx), 1, figsize=(10, 2.2*len(test_idx)))
for ax, i, r in zip(axes, test_idx, recon):
    truth = uvvis_norm[i]
    pseudo_t = model_cvae.generate_latent_mu(torch.from_numpy(truth).unsqueeze(0).unsqueeze(0).float().to(device))
    pseudo_recon = model_cvae.decode_latent_mu(pseudo_t).cpu().detach().numpy().squeeze(1).T
    ax.plot(truth, label='true', lw=1)
    ax.plot(pseudo_recon, label='recon', lw=1)
    ax.plot(r, label='pred', lw=1, alpha=0.8)
    ax.set_title(f"shot {i}")
    ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Save the trained CVAE1D + WRF artifact.
import time, pickle, pathlib

time_stamp = time.strftime("%Y%m%d_%H%M%S")
out_dir = pathlib.Path('../../model/UVVis')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"UVVis_E338_16266_16269_CVAE1D{NCOMP}+WRF_{time_stamp}.pkl"

data_to_save = {
    'model': rf_model,
    'image_model': model_cvae,
    'x_scaler': x_scaler,
    'iz_scaler': iz_scaler,
    'var_importance': rf_result['feature_importances'],
    'raw_training_set': predictors,
    'train_indices': rf_result['train_indices'],
    'test_indices': rf_result['test_indices'],
    'architecture': {
        'ncomp': NCOMP,
        'input_length': L,
        'type': 'CVAE1D Random Forest v2026.06',
    },
}
with open(out_path, 'wb') as f:
    pickle.dump(data_to_save, f)
print(f"Saved -> {out_path}")

## Train VAE1D (dense MLP) + Random Forest on UVVis spectra

Same routine as the CVAE1D section above, but swaps in the plain dense `VAE1D`
from `Python_Functions.vae`. CVAE1D trains, but reconstructions are mediocre on
these smooth, low-frequency UVVis spectra. The MLP variant is selected by
passing `model_kind='vae'` to `encode_with_cvae` — the rest of the pipeline
(per-shot normalization, RandomForest mapping, evaluation plot, save) is
unchanged so latent codes are directly comparable.

In [ ]:
from Python_Functions.training import encode_with_cvae, train_random_forest_predictor

# Reuse `uvvis_norm` prepared in the CVAE1D section: per-shot normalized,
# (N, 256) float32. Latent dim matches CVAE1D for an apples-to-apples comparison.
NCOMP_VAE = NCOMP
model_vae, latent_z_vae = encode_with_cvae(
    uvvis_norm,
    latent_dim=NCOMP_VAE,
    n_epochs=10,
    batch_size=32,
    lr=1e-3,
    model_kind='vae',
    model_kwargs={'hidden_dims': (512, 256, 128)},
)
print(f"VAE1D latent codes shape: {latent_z_vae.shape}")

In [ ]:
# Predictors → VAE1D latent codes via weighted RandomForest.
predictors_vae = predictor_tmp_simplified[sel]
print(f"Predictors: {predictors_vae.shape}, targets: {latent_z_vae.shape}")

rf_result_vae = train_random_forest_predictor(
    predictors_vae,
    latent_z_vae,
    n_estimators=500,
    max_depth=15,
    min_samples_leaf=2,
    test_size=0.2,
    weight_strategy='importance_weighted',
    kde_bandwidth=0.1,
    random_state=42,
    predictor_names=all_predictor_names_simplified
)

rf_model_vae = rf_result_vae['model']
x_scaler_vae = rf_result_vae['x_scaler']
iz_scaler_vae = rf_result_vae['iz_scaler']
print(f"[VAE1D] Train R²: {rf_result_vae['train_r2']*100:.2f}%   Test R²: {rf_result_vae['test_r2']*100:.2f}%")

In [ ]:
# Visual sanity check: predicted vs. true UVVis for a few test shots (VAE1D).
import torch

test_idx_vae = rf_result_vae['test_indices'][:6]

x_test_scaled_vae = x_scaler_vae.transform(predictors_vae[test_idx_vae])
latent_pred_vae = iz_scaler_vae.inverse_transform(rf_model_vae.predict(x_test_scaled_vae))

device = next(model_vae.parameters()).device
model_vae.eval()
with torch.no_grad():
    latent_t = torch.from_numpy(latent_pred_vae).float().to(device)
    recon = model_vae.decode_latent_mu(latent_t).cpu().numpy().squeeze(1)

fig, axes = plt.subplots(len(test_idx_vae), 1, figsize=(10, 2.2*len(test_idx_vae)))
for ax, i, r in zip(axes, test_idx_vae, recon):
    truth = uvvis_norm[i]
    pseudo_t = model_vae.generate_latent_mu(torch.from_numpy(truth).unsqueeze(0).unsqueeze(0).float().to(device))
    pseudo_recon = model_vae.decode_latent_mu(pseudo_t).cpu().detach().numpy().squeeze(1).T
    ax.plot(truth, label='true', lw=1)
    ax.plot(pseudo_recon, label='recon', lw=1)
    ax.plot(r, label='pred', lw=1, alpha=0.8)
    ax.set_title(f"shot {i}")
    ax.legend(loc='upper right', fontsize=8)
plt.suptitle('VAE1D (MLP) reconstructions', y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
# Save the trained VAE1D + WRF artifact.
import time, pickle, pathlib

time_stamp = time.strftime("%Y%m%d_%H%M%S")
out_dir = pathlib.Path('../../model/UVVis')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"UVVis_E338_16266_16269_VAE1D{NCOMP_VAE}+WRF_{time_stamp}.pkl"

data_to_save = {
    'model': rf_model_vae,
    'image_model': model_vae,
    'x_scaler': x_scaler_vae,
    'iz_scaler': iz_scaler_vae,
    'var_importance': rf_result_vae['feature_importances'],
    'raw_training_set': predictors_vae,
    'train_indices': rf_result_vae['train_indices'],
    'test_indices': rf_result_vae['test_indices'],
    'architecture': {
        'ncomp': NCOMP_VAE,
        'input_length': uvvis_norm.shape[1],
        'type': 'VAE1D Random Forest v2026.06',
    },
}
with open(out_path, 'wb') as f:
    pickle.dump(data_to_save, f)
print(f"Saved -> {out_path}")